In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

**Read data**

*Game Responses*

In [2]:
df_responses = pd.read_csv("../data/human_alignment_data/original/group_B.csv")
df_responses_level_A = pd.read_csv("../data/human_alignment_data/original/group_A.csv")
df_responses_level_C = pd.read_csv("../data/human_alignment_data/original/group_C.csv")
df_responses_level_BP = pd.read_csv("../data/human_alignment_data/original/group_BP.csv")
df_responses_calibration = pd.read_csv("../data/human_alignment_data/original/group_B_calibration.csv")

*Separate attention tests*

In [3]:
df_attention = df_responses[df_responses['game_id']<0]
df_responses = df_responses[df_responses['game_id']>=0]

df_attention_level_A = df_responses_level_A[df_responses_level_A['game_id']<0]
df_responses_level_A = df_responses_level_A[df_responses_level_A['game_id']>=0]

df_attention_level_C = df_responses_level_C[df_responses_level_C['game_id']<0]
df_responses_level_C = df_responses_level_C[df_responses_level_C['game_id']>=0]

df_attention_level_BP = df_responses_level_BP[df_responses_level_BP['game_id']<0]
df_responses_level_BP = df_responses_level_BP[df_responses_level_BP['game_id']>=0]

df_attention_calibration = df_responses_calibration[df_responses_calibration['game_id']<0]
df_responses_calibration = df_responses_calibration[df_responses_calibration['game_id']>=0]

***Filter participants with 2 or more mistakes in attention test***

Mistake -> human_conf is more than one standard deviation from mean response for the attention test

In [4]:
def filter_by_std(df, mult=1, compare="lower"):

    lower = df["human_conf"].mean() - mult*df["human_conf"].std()
    upper = df["human_conf"].mean() + mult*df["human_conf"].std()
    # print(lower,upper)
    return df[(df["human_conf"]<lower) | (df["human_conf"]>upper)]

all_attention = pd.concat([df_attention,df_attention_level_A,df_attention_level_C,df_attention_level_BP])

df_failed_attention = pd.concat([filter_by_std(all_attention[all_attention["true_prob"]==0], compare="upper"),
                                 filter_by_std(all_attention[all_attention["true_prob"]==100]),
                                 filter_by_std(all_attention[all_attention["true_prob"]==60])]).groupby(["level_name","participant_id"])["game_id"].count().reset_index()
ids=df_failed_attention[df_failed_attention["game_id"]>1]["participant_id"]


*Number of participants that failed attention check per group*

In [5]:
# print number of participants that failed attention check per group
all_data = pd.concat([df_responses,df_responses_level_A,df_responses_level_C, df_responses_level_BP])
all_data[(all_data["participant_id"].isin(ids))].groupby(["level_name"])["participant_id"].nunique()

level_name
level_A     15
level_B     17
level_BP     9
level_C     20
Name: participant_id, dtype: int64

In [6]:

df_failed_attention_cal = pd.concat([filter_by_std(df_attention_calibration[df_attention_calibration["true_prob"]==0], compare="upper"),
                                 filter_by_std(df_attention_calibration[df_attention_calibration["true_prob"]==100]),
                                 filter_by_std(df_attention_calibration[df_attention_calibration["true_prob"]==60])]).groupby(["level_name","participant_id"])["game_id"].count().reset_index()
ids_cal=df_failed_attention_cal[df_failed_attention_cal["game_id"]>1]["participant_id"]

df_responses_calibration[(df_responses_calibration["participant_id"].isin(ids_cal))].groupby(["level_name"])["participant_id"].nunique()

level_name
level_B_cal    42
Name: participant_id, dtype: int64

*Filter out participants*

In [7]:
df_responses = df_responses[~df_responses["participant_id"].isin(ids)]
df_responses_level_A = df_responses_level_A[~df_responses_level_A["participant_id"].isin(ids)]
df_responses_level_C = df_responses_level_C[~df_responses_level_C["participant_id"].isin(ids)]
df_responses_level_BP = df_responses_level_BP[~df_responses_level_BP["participant_id"].isin(ids)]

df_responses_calibration = df_responses_calibration[~df_responses_calibration["participant_id"].isin(ids_cal)]

*Save filtered out ids to use in survey analysis*

In [8]:
ids.to_csv("../data/human_alignment_data/processed/filtered_ids.csv", index=False)
ids_cal.to_csv("../data/human_alignment_data/processed/filtered_ids_cal.csv", index=False)

**Discretize Human Confidence**

In [9]:
conf_levels = ["very low", "low", "high", "very high","very high"]

def add_dm_conf(data): 
    divider = 100/(len(conf_levels)-1)
    data["dm_conf"] = np.array([conf_levels[int(i)] for i in np.floor(data["human_conf"]/divider)])
    #set dm_conf to high if human_conf is 50 and initial_decision is Red
    data.loc[(data["human_conf"]==50) & (data["initial_decision"]=="Red"),"dm_conf"]= "high"
    #set dm_conf to low if human_conf is 50 and initial_decision is Black
    data.loc[(data["human_conf"]==50) & (data["initial_decision"]=="Black"),"dm_conf"]= "low"

add_dm_conf(df_responses)
add_dm_conf(df_responses_level_A)
add_dm_conf(df_responses_level_C)
add_dm_conf(df_responses_level_BP)
add_dm_conf(df_responses_calibration)


***Table - Alignment Metrics***

In [10]:
def add_dm_conf_numeric(df):
    dm_conf_dict = {"very low": 0, "low":1 , "high":2, "very high":3}
    df["dm_conf_numeric"] = df.apply(lambda x: dm_conf_dict[x["dm_conf"]], axis=1)

add_dm_conf_numeric(df_responses)
add_dm_conf_numeric(df_responses_level_A)
add_dm_conf_numeric(df_responses_level_C)
add_dm_conf_numeric(df_responses_level_BP)
add_dm_conf_numeric(df_responses_calibration)

In [11]:
#computes expected and average alignment error
def check_alignment(data, min_datapoints=15):
        #compute max and average alignment violations
        max_aligment = 0.0
        sum_aligment = 0.0
        num_summants = 0.0
        disaligned_cells = set({})


        df_cell_prob = data.pivot_table(index='dm_conf_numeric', 
                        columns=['AI_conf'], aggfunc='mean', fill_value=0, values="true_prob",dropna=False)#.unstack()#.reset_index().fillna(value=0).rename(columns={0:'count'})
        df_cell_prob = df_cell_prob/100
        # print(df_cell_prob)

        df_cell_mass = data.pivot_table(index='dm_conf_numeric', 
                        columns=['AI_conf'], aggfunc='count', fill_value=0, values="true_prob",dropna=False)#.unstack()#.reset_index().fillna(value=0).rename(columns={0:'count'})
        # print(df_cell_mass)

        for h in range(df_cell_prob.columns.shape[0]):
            for b in range(df_cell_prob.index.shape[0]):
                for h1 in range(h+1):
                    for b1 in range(b+1):
                        # check misalignment of the pair of cells if enough datapoints in each cells
                        if (df_cell_mass.iat[b,h]>=min_datapoints) & (df_cell_mass.iat[b1,h1]>=min_datapoints):
                            alignment = max(0.0, df_cell_prob.iat[b1,h1] - df_cell_prob.iat[b,h] )
                            max_aligment = max(max_aligment, alignment)
                            num_summants +=1
                            if alignment > 0.0:
                                sum_aligment += alignment
                                disaligned_cells |= {(df_cell_prob.index[b],df_cell_prob.columns[h]),(df_cell_prob.index[b1],df_cell_prob.columns[h1])}
                                
        if num_summants > 0: 
            avg_alignment = sum_aligment / num_summants
        else:
            avg_alignment = 0.0 
        
        mae = max_aligment 
        eae = avg_alignment

        return(round(eae,5), round(mae,2))

In [12]:
df_alignment = pd.DataFrame([{"level_name": "Group B", "EAE": check_alignment(df_responses)[0],"MAE": check_alignment(df_responses)[1]}, {"level_name": "Group A", "EAE": check_alignment(df_responses_level_A)[0],"MAE": check_alignment(df_responses_level_A)[1]}, {"level_name": "Group C", "EAE": check_alignment(df_responses_level_C)[0],"MAE": check_alignment(df_responses_level_C)[1]}, {"level_name": "Group BP", "EAE": check_alignment(df_responses_level_BP)[0],"MAE": check_alignment(df_responses_level_BP)[1]}])#, {"level_name": "level_B_cal", "EAE": check_alignment(df_responses_calibration)[0],"MAE": check_alignment(df_responses_calibration)[1]}])
df_alignment

,level_name,EAE,MAE
0,Group B,0.00693,0.20
1,Group A,0.00065,0.10
2,Group C,0.00030,0.06
3,Group BP,0.00236,0.12


### Save Preprocessed Data

In [13]:
df_responses.head(2)

,participant_id,level_name,game_batch,game_id,trial_id,human_conf,human_AI_conf,AI_conf,Ai_calibrated_conf,true_prob,nr_reds_shown,initial_decision,final_decision,outcome,win,points,dm_conf,dm_conf_numeric
2,669ea18f9de24,level_B,1,45,2,90,89,85,NaN,96.923077,19,Red,Red,Red,1,3,very high,3
3,669ea18f9de24,level_B,1,41,3,77,75,69,NaN,81.538462,12,Red,Red,Red,1,4,very high,3


In [14]:
columns = ["human_conf", "dm_conf", "AI_conf", "true_prob"]
columns_cal = ["human_conf", "dm_conf", "Ai_calibrated_conf", "true_prob"]

In [15]:
df_responses[columns].to_csv("../data/human_alignment_data/processed/group_B.csv", index=False)
df_responses_level_A[columns].to_csv("../data/human_alignment_data/processed/group_A.csv", index=False)
df_responses_level_C[columns].to_csv("../data/human_alignment_data/processed/group_C.csv", index=False)
df_responses_level_BP[columns].to_csv("../data/human_alignment_data/processed/group_BP.csv", index=False)
df_responses_calibration[columns_cal].rename(columns={"Ai_calibrated_conf": "AI_conf"}).to_csv("../data/human_alignment_data/processed/group_B_calibration.csv", index=False)